In [3]:
import requests
import json
import time
import random
import os
import re


def baidu_image_spider(keyword, download_num, save_dir='./img'):
    """
    百度图片爬虫函数
    :param keyword: 搜索关键词，如 '猫咪'
    :param download_num: 想要下载的图片数量
    :param save_dir: 图片保存目录
    """

    # 创建保存目录
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 请求头
    header = {
        'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Mobile Safari/537.36',
        'Accept': 'application/json, text/plain, */*'
    }

    # 用于去重的集合
    downloaded_urls = set()
    downloaded_count = 400
    page_size = 60  # 百度每页返回的大概数量
    pn = 0  # 起始页码

    def extract_urls_by_regex(text: str):
        # 兼容 JSON 中的转义斜杠 \/
        candidates = []
        for key in ["thumbURL", "middleURL", "objURL"]:
            candidates += re.findall(rf'"{key}":"(http[^"\\]+)"', text)
            candidates += [u.replace('\\/', '/') for u in re.findall(rf'"{key}":"(http[^"\\]+)"', text)]
        # 去重且保持顺序
        seen = set()
        ordered = []
        for u in candidates:
            if u not in seen:
                seen.add(u)
                ordered.append(u)
        return ordered

    while downloaded_count < download_num:
        # 使用 params 构造查询，自动编码，避免长关键词导致的 URL 组装错误
        params = {
            'tn': 'resultjson_com',
            'ipn': 'rj',
            'ct': '201326592',
            'fp': 'result',
            'queryWord': keyword,
            'word': keyword,
            'cl': '2',
            'lm': '-1',
            'ie': 'utf-8',
            'oe': 'utf-8',
            'pn': pn,
            'rn': page_size,
            'gsm': '3c'
        }

        try:
            print(f"正在请求第 {pn // page_size + 1} 页...")
            response = requests.get(
                'https://image.baidu.com/search/acjson',
                headers=header,
                params=params,
                timeout=10
            )
            response.encoding = 'utf-8'
            data = response.text

            image_items = None
            try:
                # 优先尝试标准 JSON 解析
                obj = response.json()
                image_items = obj.get('data', []) if isinstance(obj, dict) else []
            except Exception as e:
                print(f"JSON解析错误: {e}")
                # 降级：正则从文本中抽取 URL，避免因非法转义导致的失败
                urls = extract_urls_by_regex(data)
                if not urls:
                    print("无法从响应中提取图片URL，结束。")
                    break
                # 转为统一的数据结构，便于后续处理
                image_items = [{"thumbURL": u} for u in urls]

            if not image_items:
                print("没有更多图片了")
                break

            print(f"本页获取到 {len(image_items)} 条记录")

            for item in image_items:
                if downloaded_count >= download_num:
                    break

                # 获取图片URL（优先使用thumbURL）
                thumbURL = item.get('thumbURL') or item.get('middleURL') or item.get('objURL')

                if not thumbURL:
                    continue

                # 规整可能的转义
                thumbURL = thumbURL.replace('\\/', '/')

                if thumbURL in downloaded_urls:
                    continue
                downloaded_urls.add(thumbURL)

                try:
                    # 下载图片
                    print(f"正在下载第 {downloaded_count + 1} 张图片...")
                    responseIMG = requests.get(thumbURL, headers=header, timeout=15)

                    if responseIMG.status_code == 200:
                        # 生成唯一文件名
                        lower_url = thumbURL.lower()
                        if '.webp' in lower_url or 'webp' in lower_url:
                            file_ext = '.webp'
                        elif '.png' in lower_url or 'png' in lower_url:
                            file_ext = '.png'
                        elif '.gif' in lower_url or 'gif' in lower_url:
                            file_ext = '.gif'
                        else:
                            file_ext = '.jpg'

                        filename = f"{downloaded_count + 1}{file_ext}"
                        filepath = os.path.join(save_dir, filename)

                        with open(filepath, 'wb') as f:
                            f.write(responseIMG.content)

                        downloaded_count += 1

                        # 随机延迟，避免请求过快（如需要再开启）
                        # time.sleep(random.uniform(0.5, 1.5))

                    else:
                        print(f"下载失败，状态码: {responseIMG.status_code}")

                except Exception as e:
                    print(f"下载图片时出错: {e}")
                    continue

        except Exception as e:
            print(f"请求错误: {e}")
            break

        # 翻到下一页
        pn += page_size

        # 页面间延迟
        time.sleep(random.uniform(1, 2))

    print(f"\n下载完成！共下载 {downloaded_count} 张图片到目录: {save_dir}")


# 使用示例
if __name__ == "__main__":
    # 在这里设置你的参数
    search_keyword = "派大星章鱼哥"  # 搜索关键词
    desired_count = 500     # 想要下载的图片数量
    save_directory = "./image_set/sponge_bob_images"  # 保存目录

    # 开始爬取
    baidu_image_spider(search_keyword, desired_count, save_directory)

正在请求第 1 页...
JSON解析错误: Invalid \escape: line 229 column 147 (char 183528)
本页获取到 174 条记录
正在下载第 401 张图片...
JSON解析错误: Invalid \escape: line 229 column 147 (char 183528)
本页获取到 174 条记录
正在下载第 401 张图片...
正在下载第 402 张图片...
正在下载第 402 张图片...
正在下载第 403 张图片...
正在下载第 403 张图片...
正在下载第 404 张图片...
正在下载第 405 张图片...
正在下载第 404 张图片...
正在下载第 405 张图片...
正在下载第 406 张图片...
正在下载第 407 张图片...
正在下载第 406 张图片...
正在下载第 407 张图片...
正在下载第 408 张图片...
正在下载第 409 张图片...
正在下载第 408 张图片...
正在下载第 409 张图片...
正在下载第 410 张图片...
正在下载第 410 张图片...
正在下载第 411 张图片...
正在下载第 412 张图片...
正在下载第 411 张图片...
正在下载第 412 张图片...
正在下载第 413 张图片...
正在下载第 414 张图片...
正在下载第 413 张图片...
正在下载第 414 张图片...
正在下载第 415 张图片...
正在下载第 416 张图片...
正在下载第 415 张图片...
正在下载第 416 张图片...
正在下载第 417 张图片...
正在下载第 417 张图片...
正在下载第 418 张图片...
正在下载第 418 张图片...
正在下载第 419 张图片...
正在下载第 420 张图片...
正在下载第 419 张图片...
正在下载第 420 张图片...
正在下载第 421 张图片...
正在下载第 422 张图片...
正在下载第 421 张图片...
正在下载第 422 张图片...
正在下载第 423 张图片...
正在下载第 423 张图片...
正在下载第 424 张图片...
正在下载第 425 张图片...
正在下载第 424 张图片...
正在下载